# 🛰️ Classificação Automática de Modulação (AMC)
## Comparativo: CNN × SVM × KNN — Dataset RadioML2016.10a

---

**Objetivo:** Avaliar e comparar o desempenho de três pipelines de aprendizado de máquina para identificação automática de esquemas de modulação em sinais de rádio, simulando condições reais de canal (AWGN, desequilíbrio I/Q, CFO e desvanecimento).

**Dataset:** RadioML2016.10a — 11 classes de modulação, SNR de -20 dB a +18 dB, 128 amostras complexas por segmento.

---
## Instalação e Importação de Dependências

In [12]:
import os, sys, json, time, pickle, platform, subprocess, warnings
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

CONFIG = {
    'seed': 42, 'classes': ['AM-DSB','WBFM','BPSK','QPSK','QAM16'],
    'snrs': list(range(-20, 19, 2)), 'train_ratio': 0.75,
    'cnn_filters': [60,60,60], 'cnn_kernels': [7,5,5], 'cnn_dropout': 0.3,
    'cnn_l2': 1e-4, 'cnn_lr': 1e-3, 'cnn_epochs': 60,
    'cnn_batch_size': 256, 'cnn_validation_split': 0.15,
    'knn_k': 7, 'knn_metric': 'euclidean', 'svm_c': 1.0, 'svm_max_iter': 3000,
    'cv_folds': 5, 'warmup': 2, 'runs': 10, 'batch_measure_n': 1024,
    'single_measure_n': 32, 'predict_batch_size': 256,
    'run_cross_validation': True, 'require_gpu': False,
    'gpu_train_separately': True, 'prediction_mismatch_tolerance': 0.001,
}
np.random.seed(CONFIG['seed'])
tf.keras.utils.set_random_seed(CONFIG['seed'])
# Falha explicitamente se alguma operação solicitada não puder ser colocada no dispositivo.
tf.config.set_soft_device_placement(True)
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass
ROOT = Path.cwd()
DATASET = ROOT / 'data' / 'RML2016_10a_dict.pkl'
OUT = ROOT / 'outputs' / 'amc_cpu_comparavel'
OUT.mkdir(parents=True, exist_ok=True)
CPU = '/CPU:0'; GPU = '/GPU:0'
if not DATASET.is_file(): raise FileNotFoundError(f'Dataset esperado: {DATASET}')
if CONFIG['require_gpu'] and not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('GPU obrigatória, mas não detectada pelo TensorFlow')
print('TensorFlow:', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0 | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Ambiente e dados

In [2]:
def safe_version(name):
    try: return version(name)
    except PackageNotFoundError: return None

def probe(cmd):
    try: return subprocess.check_output(cmd, text=True, stderr=subprocess.DEVNULL, timeout=5).strip()
    except (FileNotFoundError, subprocess.SubprocessError): return None

def system_info():
    try:
        import psutil
        ram = psutil.virtual_memory().total
    except ImportError:
        ram = None
        if Path('/proc/meminfo').exists():
            ram = int(Path('/proc/meminfo').read_text().split('MemTotal:')[1].split()[0]) * 1024
    build = tf.sysconfig.get_build_info()
    return dict(os=platform.platform(), python=sys.version.split()[0], numpy=np.__version__,
        sklearn=sklearn.__version__, tensorflow=tf.__version__, keras=safe_version('keras'),
        cpu=platform.processor() or probe(['lscpu']), ram_bytes=ram,
        logical_cpus=os.cpu_count(), env_threads={k:os.getenv(k) for k in
        ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','TF_NUM_INTRAOP_THREADS','TF_NUM_INTEROP_THREADS')},
        gpu_physical=[str(x) for x in tf.config.list_physical_devices('GPU')],
        gpu_logical=[str(x) for x in tf.config.list_logical_devices('GPU')],
        cuda=build.get('cuda_version'), cudnn=build.get('cudnn_version'),
        nvidia_smi=probe(['nvidia-smi','--query-gpu=name,driver_version','--format=csv,noheader']))

TIMES = {}
t0=time.perf_counter()
with DATASET.open('rb') as f: Xd=pickle.load(f, encoding='latin1')
TIMES['dataset_load_s']=time.perf_counter()-t0

def select_dataset(data, classes, snrs):
    blocks=[]; labels=[]; levels=[]
    for cls in classes:
        for snr in snrs:
            key=(cls,snr)
            if key not in data: raise KeyError(f'Combinação ausente: {key}')
            block=np.asarray(data[key], dtype=np.float32)
            if block.ndim!=3 or block.shape[1:]!=(2,128): raise ValueError((key,block.shape))
            blocks.append(block); labels.extend([cls]*len(block)); levels.extend([snr]*len(block))
    le=LabelEncoder().fit(classes)
    return np.concatenate(blocks), le.transform(labels), np.asarray(levels), le

t0=time.perf_counter()
X,y,snr,le=select_dataset(Xd,CONFIG['classes'],CONFIG['snrs'])
TIMES['select_s']=time.perf_counter()-t0
del Xd
assert len(X)==100_000, f'Esperadas 100000 amostras; encontradas {len(X)}'
t0=time.perf_counter()
idx_train,idx_test=train_test_split(np.arange(len(X)),train_size=CONFIG['train_ratio'],
    stratify=np.array([f'{a}_{b}' for a,b in zip(y,snr)]),random_state=CONFIG['seed'])
TIMES['split_s']=time.perf_counter()-t0
assert len(idx_train)==75000 and len(idx_test)==25000
assert len(np.unique(np.r_[idx_train,idx_test]))==len(X)
np.savez_compressed(OUT/'split_indices.npz',train=idx_train,test=idx_test,
    labels=y,snr=snr,classes=le.classes_)
X_train,X_test=X[idx_train],X[idx_test]
y_train,y_test=y[idx_train],y[idx_test]
snr_test=snr[idx_test]
del X
print('Treino/teste:',len(X_train),len(X_test),'| Classes:',list(le.classes_))

Treino/teste: 75000 25000 | Classes: [np.str_('AM-DSB'), np.str_('BPSK'), np.str_('QAM16'), np.str_('QPSK'), np.str_('WBFM')]


## Preparação, modelos e validação

In [3]:
def cnn_input(iq):
    return np.ascontiguousarray(np.transpose(iq,(0,2,1)),dtype=np.float32)

def order_statistics(iq):
    i,q=iq[:,0,:],iq[:,1,:]
    arrays=[np.hypot(i,q),np.arctan2(q,i),i,q]
    return np.concatenate([np.sort(a/(a.std(axis=1,keepdims=True)+1e-8),axis=1)
                           for a in arrays],axis=1).astype(np.float32,copy=False)

t0=time.perf_counter(); X_train_cnn=cnn_input(X_train); X_test_cnn=cnn_input(X_test)
TIMES['cnn_prepare_train_test_s']=time.perf_counter()-t0
t0=time.perf_counter(); X_train_os=order_statistics(X_train)
TIMES['order_statistics_train_s']=time.perf_counter()-t0
t0=time.perf_counter(); X_test_os=order_statistics(X_test)
TIMES['order_statistics_test_s']=time.perf_counter()-t0
assert X_train_os.shape==(75000,512)

def build_cnn():
    inp=keras.Input(shape=(128,2),name='iq_input'); z=inp
    for f,k in zip(CONFIG['cnn_filters'],CONFIG['cnn_kernels']):
        z=layers.Conv1D(f,k,padding='same',activation='relu',
                        kernel_regularizer=regularizers.l2(CONFIG['cnn_l2']))(z)
        z=layers.BatchNormalization()(z)
    z=layers.GlobalAveragePooling1D()(z)
    z=layers.Dense(128,activation='relu')(z)
    z=layers.Dropout(CONFIG['cnn_dropout'])(z)
    z=layers.Dense(64,activation='relu')(z)
    out=layers.Dense(len(le.classes_),activation='softmax',name='output')(z)
    model=keras.Model(inp,out,name='AMC_CNN')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=CONFIG['cnn_lr']),
                  loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model

def train_cnn(device):
    with tf.device(device): model=build_cnn()
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss',patience=10,
        restore_best_weights=True),keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',factor=0.5,patience=5,min_lr=1e-5)]
    with tf.device(device):
        start=time.perf_counter()
        hist=model.fit(X_train_cnn,y_train,epochs=CONFIG['cnn_epochs'],
            batch_size=CONFIG['cnn_batch_size'],validation_split=CONFIG['cnn_validation_split'],
            callbacks=callbacks,verbose=1,shuffle=True)
        elapsed=time.perf_counter()-start
    h=hist.history
    summary={'seconds':elapsed,'epochs':len(h['loss']),
        'best_epoch':int(np.argmin(h['val_loss']))+1,
        'training_loss':h['loss'][-1], 'validation_loss':h['val_loss'][-1],
        'training_accuracy':h['accuracy'][-1], 'validation_accuracy':h['val_accuracy'][-1]}
    pd.DataFrame(h).to_csv(OUT/f'cnn_history_{device.strip("/").lower().replace(":", "")}.csv',index=False)
    return model,summary

cnn_cpu,train_cnn_cpu=train_cnn(CPU)
knn=Pipeline([('scaler',StandardScaler()),('knn',KNeighborsClassifier(
    n_neighbors=CONFIG['knn_k'],metric=CONFIG['knn_metric'],weights='uniform',
    algorithm='brute',n_jobs=-1))])
svm=Pipeline([('scaler',StandardScaler()),('svm',LinearSVC(
    C=CONFIG['svm_c'],max_iter=CONFIG['svm_max_iter'],random_state=CONFIG['seed']))])
# O custo do scaling é registrado separadamente; o ajuste final do pipeline também o inclui.
t0=time.perf_counter(); standalone_scaler=StandardScaler().fit(X_train_os)
TIMES['scaling_fit_train_s']=time.perf_counter()-t0
t0=time.perf_counter(); standalone_scaler.transform(X_test_os)
TIMES['scaling_transform_test_s']=time.perf_counter()-t0
del standalone_scaler
TRAIN={'CNN_CPU':train_cnn_cpu}
for name,model in [('KNN_CPU',knn),('LinearSVC_CPU',svm)]:
    start=time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always'); model.fit(X_train_os,y_train)
    TRAIN[name]={'seconds':time.perf_counter()-start,'warnings':[str(w.message) for w in caught]}
    print(name,'fit:',TRAIN[name])

CV={}
if CONFIG['run_cross_validation']:
    folds=StratifiedKFold(n_splits=CONFIG['cv_folds'],shuffle=True,random_state=CONFIG['seed'])
    for name,model in [('KNN_CPU',knn),('LinearSVC_CPU',svm)]:
        start=time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            scores=cross_val_score(model,X_train_os,y_train,cv=folds,scoring='accuracy',n_jobs=1)
        CV[name]={'fold_scores':scores.tolist(),'mean':float(scores.mean()),
                  'std':float(scores.std()),'seconds':time.perf_counter()-start,
                  'warnings':[str(w.message) for w in caught]}
    print('CV:',CV)

I0000 00:00:1790507915.670142  243977 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10155 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:2b:00.0, compute capability: 7.5


Epoch 1/60


W0000 00:00:1790507916.308184  243977 cpu_allocator_impl.cc:82] Allocation of 65280000 exceeds 10% of free system memory.
I0000 00:00:1790507918.970735  244291 service.cc:153] XLA service 0x7ca2c0017be0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790507918.970774  244291 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1790507919.314448  244291 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790507923.166207  244291 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
W0000 00:00:1790507923.176626  244291 cpu_allocator_impl.cc:82] Allocation of 55572624 exceeds 10% of free system memory.


  3/250 ━━━━━━━━━━━━━━━━━━━━ 20s 82ms/step - accuracy: 0.2279 - loss: 1.7055

W0000 00:00:1790507923.534152  244293 cpu_allocator_impl.cc:82] Allocation of 55572624 exceeds 10% of free system memory.
W0000 00:00:1790507923.625260  244291 cpu_allocator_impl.cc:82] Allocation of 55572624 exceeds 10% of free system memory.
W0000 00:00:1790507923.698086  244293 cpu_allocator_impl.cc:82] Allocation of 55572624 exceeds 10% of free system memory.


250/250 ━━━━━━━━━━━━━━━━━━━━ 27s 79ms/step - accuracy: 0.5368 - loss: 0.9610 - val_accuracy: 0.2764 - val_loss: 1.9626 - learning_rate: 0.0010
Epoch 2/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 77ms/step - accuracy: 0.6094 - loss: 0.7984 - val_accuracy: 0.2068 - val_loss: 1.8318 - learning_rate: 0.0010
Epoch 3/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 22s 90ms/step - accuracy: 0.6210 - loss: 0.7676 - val_accuracy: 0.5446 - val_loss: 0.9961 - learning_rate: 0.0010
Epoch 4/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 20s 79ms/step - accuracy: 0.6259 - loss: 0.7560 - val_accuracy: 0.5877 - val_loss: 0.8772 - learning_rate: 0.0010
Epoch 5/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 21s 83ms/step - accuracy: 0.6294 - loss: 0.7485 - val_accuracy: 0.5677 - val_loss: 0.9998 - learning_rate: 0.0010
Epoch 6/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 18s 70ms/step - accuracy: 0.6309 - loss: 0.7432 - val_accuracy: 0.6229 - val_loss: 0.7611 - learning_rate: 0.0010
Epoch 7/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 76ms/step - accuracy: 0.6309 - loss: 0.7424

## CNN na GPU

In [10]:
cnn_gpu=None
if tf.config.list_physical_devices('GPU'):
    with tf.device(GPU):
        cnn_gpu=build_cnn()
        cnn_gpu.set_weights(cnn_cpu.get_weights())
else:
    print('GPU indisponível: grupo B não será calculado.')

def cnn_scores(model,iq,device,batch_size=256):
    outputs=[]; reported_device=None
    for start in range(0,len(iq),batch_size):
        with tf.device(device):
            x=tf.convert_to_tensor(iq[start:start+batch_size],dtype=tf.float32)
            out=model(x,training=False)
        actual=out.device.upper()
        assert device.upper().strip('/') in actual, (device,out.device)
        reported_device=out.device
        outputs.append(out.numpy()) # sincroniza GPU e inclui retorno ao host
    return np.concatenate(outputs),reported_device

scores_cpu,device_cpu=cnn_scores(cnn_cpu,X_test_cnn,CPU,CONFIG['predict_batch_size'])
PRED={'CNN_CPU':scores_cpu.argmax(axis=1), 'KNN_CPU':knn.predict(X_test_os),
      'LinearSVC_CPU':svm.predict(X_test_os)}
DEVICES={'CNN_CPU':device_cpu,'KNN_CPU':'CPU (scikit-learn)',
         'LinearSVC_CPU':'CPU (scikit-learn)'}
if cnn_gpu is not None:
    scores_gpu,device_gpu=cnn_scores(cnn_gpu,X_test_cnn,GPU,CONFIG['predict_batch_size'])
    PRED['CNN_GPU']=scores_gpu.argmax(axis=1); DEVICES['CNN_GPU']=device_gpu
    mismatch=float(np.mean(PRED['CNN_CPU']!=PRED['CNN_GPU']))
    max_diff=float(np.max(np.abs(scores_cpu-scores_gpu)))
    assert mismatch<=CONFIG['prediction_mismatch_tolerance'], (
        f'CPU/GPU divergem: {mismatch:.2%} de predições; máx diferença {max_diff}')
    print('CNN CPU/GPU: divergência',mismatch,'| diferença máxima',max_diff)
print('Dispositivos observados:',DEVICES)

CNN CPU/GPU: divergência 0.0 | diferença máxima 3.7550926208496094e-06
Dispositivos observados: {'CNN_CPU': '/job:localhost/replica:0/task:0/device:CPU:0', 'KNN_CPU': 'CPU (scikit-learn)', 'LinearSVC_CPU': 'CPU (scikit-learn)', 'CNN_GPU': '/job:localhost/replica:0/task:0/device:GPU:0'}


## Métricas preditivas e complexidade

In [5]:
def estimate_cnn_macs(model):
    total=0
    for layer in model.layers:
        if isinstance(layer,layers.Conv1D):
            total+=int(layer.output.shape[1])*int(layer.output.shape[2])*int(layer.input.shape[-1])*layer.kernel_size[0]
        elif isinstance(layer,layers.Dense):
            total+=int(layer.input.shape[-1])*layer.units
    return total

def metrics_for(name,pred):
    p,r,f,_=precision_recall_fscore_support(y_test,pred,labels=np.arange(len(le.classes_)),
        average='macro',zero_division=0)
    pw,rw,fw,_=precision_recall_fscore_support(y_test,pred,labels=np.arange(len(le.classes_)),
        average='weighted',zero_division=0)
    return dict(model=name,accuracy=accuracy_score(y_test,pred),precision_macro=p,
        recall_macro=r,f1_macro=f,precision_weighted=pw,recall_weighted=rw,f1_weighted=fw)

GLOBAL=[]; PER_CLASS=[]; PER_SNR=[]; CLASS_SNR=[]
for name,pred in PRED.items():
    GLOBAL.append(metrics_for(name,pred))
    pp,rr,ff,support=precision_recall_fscore_support(y_test,pred,
        labels=np.arange(len(le.classes_)),zero_division=0)
    for j,cls in enumerate(le.classes_):
        PER_CLASS.append(dict(model=name,modulation=cls,support=int(support[j]),
                              precision=pp[j],recall=rr[j],f1=ff[j]))
    for level in CONFIG['snrs']:
        mask=snr_test==level
        PER_SNR.append(dict(model=name,snr_db=level,n=int(mask.sum()),
                            accuracy=accuracy_score(y_test[mask],pred[mask])))
        for j,cls in enumerate(le.classes_):
            sub=mask & (y_test==j)
            CLASS_SNR.append(dict(model=name,snr_db=level,modulation=cls,
                n=int(sub.sum()),accuracy=float(np.mean(pred[sub]==j)) if sub.any() else None))
    for suffix,mask in [('global',np.ones(len(y_test),bool)),('snr_10db',snr_test==10)]:
        matrix=confusion_matrix(y_test[mask],pred[mask],labels=np.arange(len(le.classes_)))
        pd.DataFrame(matrix,index=le.classes_,columns=le.classes_).to_csv(
            OUT/f'confusion_{name.lower()}_{suffix}.csv')
pd.DataFrame(GLOBAL).to_csv(OUT/'metrics_global.csv',index=False)
pd.DataFrame(PER_CLASS).to_csv(OUT/'metrics_by_class.csv',index=False)
pd.DataFrame(PER_SNR).to_csv(OUT/'accuracy_by_snr.csv',index=False)
pd.DataFrame(CLASS_SNR).to_csv(OUT/'accuracy_class_snr.csv',index=False)

n_train,dim=X_train_os.shape
macs=estimate_cnn_macs(cnn_cpu)
complexity={
 'CNN_CPU':dict(total_params=cnn_cpu.count_params(),trainable_params=sum(int(np.prod(w.shape)) for w in cnn_cpu.trainable_weights),
    estimated_macs_per_sample=macs,estimated_flops_conv_dense=2*macs,
    model_weight_bytes=sum(int(np.prod(w.shape))*np.dtype(w.dtype).itemsize for w in cnn_cpu.weights),
    scope='Conv1D + Dense; excludes BN, activations, pooling, softmax'),
 'KNN_CPU':dict(stored_samples=n_train,features=dim,
    estimated_storage_bytes=knn.named_steps['knn']._fit_X.nbytes,
    distance_terms_per_query=n_train*dim,search_order='O(ND) brute force; plus neighbor selection'),
 'LinearSVC_CPU':dict(classifiers=svm.named_steps['svm'].coef_.shape[0],features=dim,
    coefficients=int(svm.named_steps['svm'].coef_.size),
    dot_product_terms_per_query=int(svm.named_steps['svm'].coef_.size),search_order='O(MD)')}
if cnn_gpu is not None: complexity['CNN_GPU']=complexity['CNN_CPU'].copy()

## Medições

In [16]:
# ATUALIZAÇÃO 27/09/2026 — MEDIÇÃO SEM TREINAMENTO
def sampled_positions(n):
    n=min(n,len(y_test))
    pos,_=train_test_split(np.arange(len(y_test)),train_size=n,
        stratify=(np.array([f'{a}_{b}' for a,b in zip(y_test,snr_test)])
                  if n >= 2*len(le.classes_)*len(CONFIG['snrs']) else y_test),
        random_state=CONFIG['seed'])
    return np.sort(pos)

RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RESULTS=OUT/'results'/RUN_ID
for section in ('metrics','latency','complexity','cpu_gpu','matrices','figures'):
    (RESULTS/section).mkdir(parents=True,exist_ok=True)

BATCH_POS=sampled_positions(CONFIG['batch_measure_n'])
SINGLE_POS=sampled_positions(CONFIG['single_measure_n'])
np.savez_compressed(RESULTS/'benchmark_indices.npz',batch_test_positions=BATCH_POS,
                    single_test_positions=SINGLE_POS)
BENCH_INPUTS={}
for scenario,pos in [('batch',BATCH_POS),('single',SINGLE_POS)]:
    BENCH_INPUTS[scenario]={'raw':X_test[pos], 'cnn':X_test_cnn[pos],
                            'classic':X_test_os[pos]}
    for name,model in [('KNN_CPU',knn),('LinearSVC_CPU',svm)]:
        BENCH_INPUTS[scenario][name+'_scaled']=model.named_steps['scaler'].transform(
            BENCH_INPUTS[scenario]['classic'])

# 'classifier': rede CNN, ou estimador clássico com features já escaladas.
# 'feature_extraction' e 'scaling' são medidos separadamente;
# 'preprocessing' mede a combinação completa e 'complete' inclui a predição.
def predict_component(name,mode,iq=None,prepared=None):
    if name.startswith('CNN'):
        if mode in ('preprocessing','feature_extraction'):
            return cnn_input(iq)
        model,device=(cnn_cpu,CPU) if name=='CNN_CPU' else (cnn_gpu,GPU)
        values=prepared if mode=='classifier' else cnn_input(iq)
        return cnn_scores(model,values,device,len(values))[0].argmax(axis=1)
    model=knn if name=='KNN_CPU' else svm
    if mode=='feature_extraction': return order_statistics(iq)
    if mode=='scaling': return model.named_steps['scaler'].transform(prepared)
    if mode=='preprocessing':
        return model.named_steps['scaler'].transform(order_statistics(iq))
    if mode=='classifier':
        return model.named_steps['knn' if name=='KNN_CPU' else 'svm'].predict(prepared)
    return model.predict(order_statistics(iq))

def measure_run(name,pos,mode,scenario):
    step=CONFIG['predict_batch_size'] if scenario=='batch' else 1
    data=BENCH_INPUTS[scenario]
    records=[]
    for offset in range(0,len(pos),step):
        end=min(offset+step,len(pos))
        if mode=='classifier':
            key='cnn' if name.startswith('CNN') else name+'_scaled'
            prepared=data[key][offset:end]
            iq=None
        elif mode=='scaling':
            iq=None;prepared=data['classic'][offset:end]
        else:
            iq=data['raw'][offset:end];prepared=None
        start=time.perf_counter()
        result=predict_component(name,mode,iq=iq,prepared=prepared)
        sec=time.perf_counter()-start
        assert len(result)==end-offset
        if scenario=='single':
            records.append((sec,int(pos[offset])))
        else:
            records.append((sec,None))
    if scenario=='batch': return [(sum(x[0] for x in records),None)]
    return records

NAMES=['CNN_CPU','KNN_CPU','LinearSVC_CPU']+(['CNN_GPU'] if cnn_gpu is not None else [])
MODES=('feature_extraction','scaling','preprocessing','classifier','complete')
RAW=[];BENCH=[]
for scenario,pos in [('batch',BATCH_POS),('single',SINGLE_POS)]:
    for mode in MODES:
        eligible=[name for name in NAMES if not (name.startswith('CNN') and mode=='scaling')]
        for name in eligible:
            for _ in range(CONFIG['warmup']): measure_run(name,pos,mode,scenario)
        for run in range(CONFIG['runs']):
            for name in eligible[run%len(eligible):]+eligible[:run%len(eligible)]:
                for sec,position in measure_run(name,pos,mode,scenario):
                    n=1 if scenario=='single' else len(pos)
                    RAW.append(dict(run_id=RUN_ID,recorded_utc=datetime.now(timezone.utc).isoformat(),
                        model=name,device='GPU' if name=='CNN_GPU' else 'CPU',
                        scenario=scenario,mode=mode,run=run+1,
                        test_position=position,n=n,batch_size=1 if scenario=='single' else CONFIG['predict_batch_size'],
                        total_seconds=sec,ms_per_sample=1000*sec/n,throughput_samples_s=n/sec))
        print('Medição concluída:',scenario,mode)

raw_df=pd.DataFrame(RAW)
for (name,scenario,mode),group in raw_df.groupby(['model','scenario','mode'],sort=False):
    values=group.ms_per_sample.to_numpy()
    # Percentis single: distribuição por consulta; batch: distribuição entre repetições.
    run_totals=group.groupby('run',sort=True).total_seconds.sum()
    n_total=int(group.n.sum())
    BENCH.append(dict(run_id=RUN_ID,model=name,device='GPU' if name=='CNN_GPU' else 'CPU',
        scenario=scenario,mode=mode,n_samples_per_run=len(SINGLE_POS) if scenario=='single' else len(BATCH_POS),
        batch_size=1 if scenario=='single' else CONFIG['predict_batch_size'],
        runs=CONFIG['runs'],warmup=CONFIG['warmup'],observations=len(values),
        mean_ms=float(values.mean()),median_ms=float(np.median(values)),std_ms=float(values.std()),
        p95_ms=float(np.percentile(values,95)),p99_ms=float(np.percentile(values,99)),
        min_ms=float(values.min()),max_ms=float(values.max()),
        mean_total_s=float(run_totals.mean()),total_measured_s=float(run_totals.sum()),
        throughput_samples_s=float(n_total/run_totals.sum())))
raw_df.to_csv(RESULTS/'latency'/'latency_raw.csv',index=False)
pd.DataFrame(BENCH).to_csv(RESULTS/'latency'/'latency_summary.csv',index=False)

SPEEDUPS=[]
if cnn_gpu is not None:
    for scenario in ('batch','single'):
        for mode in ('classifier','complete'):
            a=next(r for r in BENCH if (r['model'],r['scenario'],r['mode'])==('CNN_CPU',scenario,mode))
            b=next(r for r in BENCH if (r['model'],r['scenario'],r['mode'])==('CNN_GPU',scenario,mode))
            SPEEDUPS.append(dict(scenario=scenario,mode=mode,
                latency_cpu_over_gpu=a['mean_ms']/b['mean_ms'],
                throughput_gpu_over_cpu=b['throughput_samples_s']/a['throughput_samples_s']))
    pd.DataFrame(SPEEDUPS).to_csv(RESULTS/'cpu_gpu'/'speedup.csv',index=False)


Medição concluída: batch feature_extraction
Medição concluída: batch scaling
Medição concluída: batch preprocessing
Medição concluída: batch classifier
Medição concluída: batch complete
Medição concluída: single feature_extraction
Medição concluída: single scaling
Medição concluída: single preprocessing
Medição concluída: single classifier
Medição concluída: single complete


## Treinamento GPU (opcional)

In [17]:
GPU_TRAIN=None
if CONFIG['gpu_train_separately']:
    if cnn_gpu is None: raise RuntimeError('Treinamento GPU solicitado, mas GPU indisponível')
    tf.keras.utils.set_random_seed(CONFIG['seed'])
    independent_model,GPU_TRAIN=train_cnn(GPU)
    independent_model.save(OUT/'cnn_gpu_independent_training.keras')
    del independent_model

Epoch 1/60


I0000 00:00:1790512800.207130  244292 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_439220__.38


249/250 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5336 - loss: 0.9629

I0000 00:00:1790512803.662879  244289 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_439220__.38


250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.5336 - loss: 0.9630 - val_accuracy: 0.2005 - val_loss: 2.0981 - learning_rate: 0.0010
Epoch 2/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6103 - loss: 0.7995 - val_accuracy: 0.2138 - val_loss: 2.9764 - learning_rate: 0.0010
Epoch 3/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6215 - loss: 0.7654 - val_accuracy: 0.4839 - val_loss: 1.3326 - learning_rate: 0.0010
Epoch 4/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6193 - loss: 0.7720 - val_accuracy: 0.4785 - val_loss: 1.3293 - learning_rate: 0.0010
Epoch 5/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6282 - loss: 0.7495 - val_accuracy: 0.2598 - val_loss: 4.3111 - learning_rate: 0.0010
Epoch 6/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6297 - loss: 0.7438 - val_accuracy: 0.5948 - val_loss: 0.8296 - learning_rate: 0.0010
Epoch 7/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6335 - loss: 0.7397 - val_accura

## Exportação e figuras

In [ ]:
# ATUALIZAÇÃO — TABELAS E FIGURAS DO ARTIGO
# Esta célula usa apenas modelos, predições e medições já presentes em memória.
# Não executa fit, cross-validation nem pesquisa de hiperparâmetros.
import shutil

def csv_out(frame,section,filename):
    path=RESULTS/section/filename
    frame.to_csv(path,index=False)
    return path

def json_out(obj,filename):
    (RESULTS/filename).write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str))

def save_figure(fig,filename):
    fig.tight_layout()
    for ext in ('pdf','png'):
        fig.savefig(RESULTS/'figures'/f'{filename}.{ext}',
                    bbox_inches='tight',dpi=220 if ext=='png' else None)
    plt.close(fig)

now=datetime.now(timezone.utc).isoformat()
training={name:details.copy() for name,details in TRAIN.items()}
if GPU_TRAIN is not None:
    training['CNN_GPU_independent_training']=GPU_TRAIN

config=dict(CONFIG)
config.update(dataset_file=DATASET.name,dataset_samples=len(y_train)+len(y_test),
    dataset_key_order='classes then snrs (CONFIG order)',train_samples=len(y_train),
    test_samples=len(y_test),created_utc=now,run_id=RUN_ID,
    split_indices_file='../../split_indices.npz',benchmark_indices_file='benchmark_indices.npz')
environment=system_info()
models_meta=[]
for name in NAMES:
    model_type='CNN' if name.startswith('CNN') else ('KNN' if name.startswith('KNN') else 'LinearSVC')
    models_meta.append(dict(run_id=RUN_ID,model=name,pipeline=model_type,
        device='GPU' if name=='CNN_GPU' else 'CPU',exported_utc=now,
        observed_device=DEVICES[name],seed=CONFIG['seed'],dataset=DATASET.name,
        classes=CONFIG['classes'],snrs=CONFIG['snrs'],
        train_samples=len(y_train),test_samples=len(y_test),
        inference_batch_size=CONFIG['predict_batch_size'],
        hyperparameters={k:v for k,v in CONFIG.items() if k.startswith(
            {'CNN':'cnn_','KNN':'knn_','LinearSVC':'svm_'}[model_type])}))
json_out(dict(run_id=RUN_ID,created_utc=now,configuration=config,
              environment=environment,models=models_meta,
              training=training,cross_validation=CV,
              devices=DEVICES,offline_preprocessing_seconds=TIMES,
              inference_definitions={
                'classifier':'CNN from prepared I/Q, or fitted estimator from pre-scaled features',
                'preprocessing':'I/Q format preparation, or feature extraction plus scaling',
                'complete':'I/Q through preprocessing and prediction',
                'batch_percentiles':'distribution of amortized ms/sample across repeated test selections',
                'single_percentiles':'distribution of measured per-query latency across all runs',
                'gpu_prediction':'CNN CPU model copied with same weights to GPU',
                'gpu_training':'independent training, possible numerical differences in weights'}),
         'experiment_metadata.json')

metrics_df=pd.DataFrame(GLOBAL).assign(run_id=RUN_ID,
    device=lambda f:np.where(f.model.eq('CNN_GPU'),'GPU','CPU'))
csv_out(metrics_df,'metrics','global_metrics.csv')
csv_out(pd.DataFrame(PER_CLASS).assign(run_id=RUN_ID),'metrics','class_metrics.csv')
csv_out(pd.DataFrame(PER_SNR).assign(run_id=RUN_ID),'metrics','accuracy_by_snr.csv')
csv_out(pd.DataFrame(CLASS_SNR).assign(run_id=RUN_ID),'metrics','accuracy_by_class_snr.csv')
snr_wide=pd.DataFrame(PER_SNR).query("model != 'CNN_GPU'").pivot(
    index='snr_db',columns='model',values='accuracy').reset_index()
snr_wide=snr_wide[['snr_db','CNN_CPU','KNN_CPU','LinearSVC_CPU']]
csv_out(snr_wide,'metrics','accuracy_by_snr_cpu_wide.csv')
np.savez_compressed(RESULTS/'metrics'/'test_predictions.npz',
    y_true=y_test,snr_db=snr_test,test_indices=idx_test,
    **{f'pred_{name}':pred for name,pred in PRED.items()})

for name,pred in PRED.items():
    for suffix,mask in [('global',np.ones(len(y_test),bool)),('snr_10db',snr_test==10)]:
        if not mask.any(): continue
        cm=confusion_matrix(y_test[mask],pred[mask],labels=np.arange(len(le.classes_)))
        normal=cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
        stem=f'{name.lower()}_{suffix}'
        pd.DataFrame(cm,index=le.classes_,columns=le.classes_).to_csv(
            RESULTS/'matrices'/f'{stem}_counts.csv',index_label='true_class')
        pd.DataFrame(normal,index=le.classes_,columns=le.classes_).to_csv(
            RESULTS/'matrices'/f'{stem}_normalized.csv',index_label='true_class')

# Estatísticas de pré-processamento usam o mesmo protocolo, seleção e lotes da inferência.
bench_df=pd.DataFrame(BENCH)
csv_out(bench_df[bench_df['mode'].isin(
    ('feature_extraction','scaling','preprocessing'))],'latency','preprocessing_summary.csv')
offline_df=pd.DataFrame([dict(run_id=RUN_ID,stage=k,seconds=v)
                         for k,v in TIMES.items()])
csv_out(offline_df,'latency','offline_preprocessing.csv')

# Persistir modelos treinados apenas se ainda não tiverem sido salvos.
if not (OUT/'cnn_cpu.keras').exists(): cnn_cpu.save(OUT/'cnn_cpu.keras')
if cnn_gpu is not None and not (OUT/'cnn_gpu_same_weights.keras').exists():
    cnn_gpu.save(OUT/'cnn_gpu_same_weights.keras')
if not (OUT/'knn_pipeline.joblib').exists(): joblib.dump(knn,OUT/'knn_pipeline.joblib')
if not (OUT/'linearsvc_pipeline.joblib').exists(): joblib.dump(svm,OUT/'linearsvc_pipeline.joblib')
if not (OUT/'label_encoder.joblib').exists(): joblib.dump(le,OUT/'label_encoder.joblib')

complexity_rows=[]
for name,values in complexity.items():
    model_disk=(OUT/('cnn_cpu.keras' if name=='CNN_CPU' else
                    'cnn_gpu_same_weights.keras' if name=='CNN_GPU' else
                    'knn_pipeline.joblib' if name=='KNN_CPU' else 'linearsvc_pipeline.joblib'))
    complexity_rows.append(dict(model=name,device='GPU' if name=='CNN_GPU' else 'CPU',
        total_params=values.get('total_params'),trainable_params=values.get('trainable_params'),
        estimated_macs_per_sample=values.get('estimated_macs_per_sample'),
        estimated_flops_conv_dense=values.get('estimated_flops_conv_dense'),
        stored_samples=values.get('stored_samples'),features=values.get('features'),
        classifiers=values.get('classifiers'),
        coef_shape=str(svm.named_steps['svm'].coef_.shape) if name=='LinearSVC_CPU' else None,
        coefficients=values.get('coefficients'),
        distance_terms_per_query=values.get('distance_terms_per_query'),
        dot_product_terms_per_query=values.get('dot_product_terms_per_query'),
        model_weight_bytes=values.get('model_weight_bytes'),
        estimated_storage_bytes=values.get('estimated_storage_bytes'),
        model_file_bytes=model_disk.stat().st_size if model_disk.is_file() else None,
        scope=values.get('scope',values.get('search_order'))))
complexity_df=pd.DataFrame(complexity_rows)
csv_out(complexity_df,'complexity','model_complexity.csv')

# Tabela completa dos cenários; latência em lote é tempo amortizado por exemplo.
csv_out(bench_df,'latency','latency_summary.csv')
csv_out(raw_df,'latency','latency_raw.csv')

def benchmark_rows(scenario,models):
    selected=bench_df[(bench_df.scenario==scenario)&(bench_df.model.isin(models))]
    main=selected[selected['mode']=='complete'].copy()
    main=main.rename(columns={c:'complete_'+c for c in
        ('mean_ms','median_ms','p95_ms','p99_ms','throughput_samples_s')})
    for mode in ('preprocessing','classifier','feature_extraction','scaling'):
        stage=selected[selected['mode']==mode][['model','mean_ms','median_ms','p95_ms','p99_ms']]
        stage=stage.rename(columns={c:mode+'_'+c for c in
            ('mean_ms','median_ms','p95_ms','p99_ms')})
        main=main.merge(stage,on='model',how='left')
    return main

cpu_order=['CNN_CPU','KNN_CPU','LinearSVC_CPU']
main=benchmark_rows('batch',cpu_order).merge(
    metrics_df.drop(columns=['run_id']),on=['model','device'],how='left')
main=main.merge(complexity_df,on=['model','device'],how='left')
main['order']=main.model.map({name:i for i,name in enumerate(cpu_order)})
main=main.sort_values('order').drop(columns=['order'])
csv_out(main,'metrics','main_cpu_comparison.csv')
latex_cols=['model','device','accuracy','f1_macro','complete_mean_ms',
            'complete_p95_ms','complete_p99_ms','complete_throughput_samples_s',
            'total_params','stored_samples','estimated_macs_per_sample']
# O CSV com estas colunas pode ser importado diretamente no fluxo LaTeX.
csv_out(main[latex_cols],'metrics','main_cpu_latex_columns.csv')

# Tempos de treinamento: a medição independente da GPU fica identificada.
train_rows=[]
for name,record in training.items():
    train_rows.append(dict(model=name,device='GPU' if 'GPU' in name else 'CPU',
                           training_seconds=record['seconds'],
                           independent_training=(name=='CNN_GPU_independent_training')))
csv_out(pd.DataFrame(train_rows),'metrics','training_times.csv')

composition=[]
for _,r in main.iterrows():
    total=r.complete_mean_ms; prep=r.preprocessing_mean_ms; classification=r.classifier_mean_ms
    denom=prep+classification
    composition.append(dict(model=r.model,scenario='batch',
        preprocessing_ms=prep,classifier_ms=classification,pipeline_complete_ms=total,
        preprocessing_fraction_of_total=prep/total,
        classification_fraction_of_total=classification/total,
        preprocessing_share_of_stages=prep/denom,
        classification_share_of_stages=classification/denom,
        nonadditivity_ms=total-denom))
composition_df=pd.DataFrame(composition)
csv_out(composition_df,'latency','latency_composition.csv')

if cnn_gpu is not None:
    gpu_table=benchmark_rows('batch',['CNN_CPU','CNN_GPU'])
    gpu_table=pd.concat([gpu_table,benchmark_rows('single',['CNN_CPU','CNN_GPU'])],ignore_index=True)
    gpu_table=gpu_table.merge(metrics_df[['model','accuracy']],on='model',how='left')
    gpu_table['training_seconds']=gpu_table.model.map({
        'CNN_CPU':train_cnn_cpu['seconds'],
        'CNN_GPU':GPU_TRAIN['seconds'] if GPU_TRAIN is not None else np.nan})
    gpu_table['training_is_independent']=gpu_table.model.eq('CNN_GPU')
    csv_out(gpu_table,'cpu_gpu','cnn_cpu_gpu.csv')
    speedup_rows=pd.DataFrame(SPEEDUPS)
    if GPU_TRAIN is not None:
        speedup_rows['training_cpu_over_gpu_independent']=train_cnn_cpu['seconds']/GPU_TRAIN['seconds']
    csv_out(speedup_rows,'cpu_gpu','speedup.csv')
    pred_difference=np.mean(PRED['CNN_CPU']!=PRED['CNN_GPU'])
    acc_difference=abs(metrics_df.set_index('model').loc['CNN_CPU','accuracy']-
                       metrics_df.set_index('model').loc['CNN_GPU','accuracy'])
    json_out(dict(prediction_mismatch_fraction=float(pred_difference),
                  accuracy_absolute_difference=float(acc_difference),
                  threshold=CONFIG['prediction_mismatch_tolerance'],
                  requires_investigation=bool(pred_difference>CONFIG['prediction_mismatch_tolerance'])),
             'cpu_gpu/prediction_equivalence.json')

# Figuras: os dados são lidos das tabelas exportadas, sem acesso ao dataset ou modelos.
plot_snr=pd.read_csv(RESULTS/'metrics'/'accuracy_by_snr_cpu_wide.csv')
fig,ax=plt.subplots(figsize=(8.4,4.4))
for name in cpu_order:
    ax.plot(plot_snr.snr_db,100*plot_snr[name],marker='o',markersize=3,label=name)
ax.set(xlabel='SNR (dB)',ylabel='Accuracy (%)',ylim=(0,100),
       xticks=CONFIG['snrs'],title='Accuracy by SNR: pipelines in CPU')
ax.grid(alpha=.25);ax.legend()
save_figure(fig,'accuracy_by_snr_cpu')

plot_main=pd.read_csv(RESULTS/'metrics'/'main_cpu_comparison.csv')
fig,ax=plt.subplots(figsize=(6.7,4.5))
for _,r in plot_main.iterrows():
    ax.scatter(r.complete_mean_ms,100*r.accuracy,s=85)
    ax.annotate(r.model,(r.complete_mean_ms,100*r.accuracy),
                xytext=(6,6),textcoords='offset points')
ax.set(xlabel='Tempo amortizado do pipeline completo (ms/amostra; lote)',
       ylabel='Acurácia (%)',title='Acurácia × latência em CPU')
ax.grid(alpha=.25);ax.margins(x=.25,y=.3)
save_figure(fig,'accuracy_vs_latency_cpu')

fig,ax=plt.subplots(figsize=(7,4.5))
ax.bar(plot_main.model,plot_main.complete_mean_ms,color=['#4C78A8','#F58518','#54A24B'])
ax.set(xlabel='Pipeline em CPU',ylabel='Tempo amortizado (ms/amostra; lote)',
       title='Latência completa de inferência')
ax.grid(axis='y',alpha=.25)
save_figure(fig,'latency_complete_cpu')

if cnn_gpu is not None:
    plot_gpu=pd.read_csv(RESULTS/'cpu_gpu'/'cnn_cpu_gpu.csv')
    fig,ax=plt.subplots(1,2,figsize=(10,4.1))
    for axis,scenario,metric,label in [
        (ax[0],'batch','complete_throughput_samples_s','Throughput (amostras/s)'),
        (ax[1],'single','complete_mean_ms','Latência por consulta (ms)')]:
        subset=plot_gpu[plot_gpu.scenario==scenario]
        axis.bar(subset.model,subset[metric],color=['#4C78A8','#E45756'])
        axis.set(ylabel=label,title=f'CNN CPU × GPU: {scenario}')
        axis.grid(axis='y',alpha=.25)
    save_figure(fig,'cnn_cpu_gpu')

plot_comp=pd.read_csv(RESULTS/'latency'/'latency_composition.csv')
fig,ax=plt.subplots(figsize=(7,4.2))
x=np.arange(len(plot_comp))
ax.bar(x,100*plot_comp.preprocessing_share_of_stages,label='Pré-processamento')
ax.bar(x,100*plot_comp.classification_share_of_stages,
       bottom=100*plot_comp.preprocessing_share_of_stages,label='Classificador')
ax.set(xticks=x,xticklabels=plot_comp.model,ylabel='Parcela das etapas medidas (%)',
       title='Composição normalizada das etapas de inferência')
ax.legend();ax.grid(axis='y',alpha=.25)
save_figure(fig,'latency_composition_cpu')

print('Resultados organizados em:',RESULTS.resolve())
print('Medições de CPU (batch, pipeline completo):')
print(main[['model','accuracy','complete_mean_ms','complete_p95_ms','complete_p99_ms',
            'complete_throughput_samples_s']].to_string(index=False))
print('A soma dos tempos isolados pode diferir do tempo do pipeline completo por efeitos de cache e overhead.')


Resultados organizados em: /home/mairon/Projetos/IFCE/AMC_KNN_CNN_LinearSVM/outputs/amc_cpu_comparavel/results/20260927T123909466457Z
Medições de CPU (batch, pipeline completo):
        model  accuracy  complete_mean_ms  complete_p95_ms  complete_p99_ms  complete_throughput_samples_s
      CNN_CPU   0.64752          0.131991         0.160234         0.165798                    7576.271713
      KNN_CPU   0.57016          0.395361         0.406899         0.410230                    2529.335919
LinearSVC_CPU   0.55260          0.008120         0.009062         0.009203                  123159.068092
A soma dos tempos isolados pode diferir do tempo do pipeline completo por efeitos de cache e overhead.
